In [0]:
--use catalog lakehouse_dev;
--use schema datasphere_test;
create or refresh streaming table customer_bronze
  comment "Raw data from customers CDC feed"
select 
*,
current_timestamp() as processing_time
from stream read_files("/Volumes/lakehouse_dev/datasphere_test/managedvolume/customers", format=>"json", schema => "customer_id string, address string, city string, state string, operation string");


In [0]:
use catalog lakehouse_dev;
use schema datasphere_test;
create or refresh streaming table customer_silver;
apply changes into LIVE.customer_silver
  from stream(LIVE.customer_bronze)
  keys (customer_id)
  apply as delete when operation = "DELETE"
  sequence by processing_time
  columns * except (operation);


  

In [0]:
create materialized view customer_gold
comment "Total active customer"
as select count(*) as CustomerCount
from LIVE.customer_silver